In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from bs4 import BeautifulSoup

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.by import By

from time import sleep

import os



In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'US CFPB' ## change to current controller name



print(f"Running {regulatorName} Web Scraping Tool v.1.2")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

Running US CFPB Web Scraping Tool v.1.2


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



# %%

In [4]:

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = {

    'US CFPB 1': 'https://www.consumerfinance.gov/compliance/supervision-examinations/institutions/',

    'US CFPB 2': 'https://www.consumerfinance.gov/compliance/supervision-examinations/institutions/',

    }



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}

Typology={
       regulatorName + ' 1': 'List of CFPB Depository Institutions',
       regulatorName + ' 2': 'List of CFPB Depository Affilliates',
}

processdate = now.strftime('%Y-%m-%d')



# %%


In [5]:
#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def click_element(driver, XPATH, Msg=None):

    for time in range(10):

        try:

            driver.find_element(By.XPATH, XPATH).click()

            if Msg!=None:

                print(f"[INFO] : - {Msg}")

            break

        except:

            print(f"[ERROR] : - Retrying {time+1}/10 to click element in this page: {XPATH} ")

            sleep(1)

    else:

        raise Exception('[ERROR] : Failed to get presence for this element in this page:')



def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : - {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : - Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : Failed to Download {fileType} file. Run Script again' )

    return  os.listdir(tempfolder)[0]



# %%

In [ ]:
for k, reg in enumerate(regdict):

	print(f"[INFO] : Working {k+1}/{len(regdict)} | {reg} ")

	driver.get(regdict[reg])

	sleep(4)

	soup = BeautifulSoup(driver.page_source, 'html.parser')
	current_links = soup.find_all('a', class_ = 'a-link',href=True)
	#print(current_links)

	sleep(5)
	for current_link in current_links:
		# print(current_link['href'])
		# print(current_link.text)
		if 'Current list Excel' in current_link.text:
			driver.get(current_link['href'])
			sleep(5)
			dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
			sleep(5)
			if reg =='US CFPB 1':
				df = pd.read_excel(dl_files[0], sheet_name=0)
			elif reg == 'US CFPB 2':
				df = pd.read_excel(dl_files[0], sheet_name=1) # Second sheet to read by index 
			sleep(5)
			df.columns = ['ID', 'Institution', 'City', 'State', 'Regulator', 'Total_Assets']
			df = df[1:]
			df = df.reset_index(drop = True)
			print(f"[INFO] : -- DataFrame '{reg}' | containe = {df.shape}")
			for index, row in df.iterrows():
				sqldict['Name'].append(row['Institution'])
				sqldict['City'].append(row['City'])
				sqldict['RegulationType'].append('Regulated')
				sqldict['InternalID_1_type'].append('ID')
				sqldict['InternalID_1'].append(row['ID'])
				sqldict["Cntry"].append(row["State"] + ' US')  
				sqldict['ListProcessDate'].append(processdate)
				sqldict['RegCtry'].append(reg.split(' ')[0])
				sqldict['RegCode'].append(reg.split(' ')[1])
				sqldict['ListCode'].append(reg.split(' ')[-1])
				sqldict['ListName'].append(Typology[reg])

			sqldict = bourange_same_length_array(sqldict)


	for del_file in os.listdir(tempfolder):

		os.remove(os.path.join(tempfolder, del_file))


		


[INFO] : Working 1/2 | US CFPB 1 
[<a class="a-link o-mega-menu__content-link o-mega-menu__content-2-link" href="/complaint/">
<svg aria-hidden="true" class="cf-icon-svg cf-icon-svg--complaint" viewbox="0 0 15 19" xmlns="http://www.w3.org/2000/svg"><path d="M14.032 5.286v7.276a1.11 1.11 0 0 1-1.108 1.108H8.75l-1.02 1.635a.273.273 0 0 1-.503 0l-1.02-1.635h-4.13a1.11 1.11 0 0 1-1.109-1.108V5.286a1.11 1.11 0 0 1 1.108-1.108h10.848a1.11 1.11 0 0 1 1.108 1.108M8.206 11.34a.706.706 0 1 0-.706.705.706.706 0 0 0 .706-.705m-1.26-1.83a.554.554 0 1 0 1.108 0V6.275a.554.554 0 1 0-1.108 0z"></path></svg>
<span class="a-link__text">Submit a Complaint</span>
</a>, <a class="a-link o-mega-menu__content-link o-mega-menu__content-2-link" href="/your-story/">
<svg aria-hidden="true" class="cf-icon-svg cf-icon-svg--open-quote" viewbox="0 0 16 19" xmlns="http://www.w3.org/2000/svg"><path d="M6.808 11.29a3.2 3.2 0 0 1 .097 1.464 3.2 3.2 0 0 1-.535 1.277 3.15 3.15 0 0 1-2.286 1.316 3.43 3.43 0 0 1-2.628-.836

In [8]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\3\ipykernel_24708\646804796.py:7: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [10]:
df.to_csv('2025_US_CFPB_v1.csv')